In [4]:
import SimpleITK as sitk
import numpy as np 

path_vessels = "/projects/vig/Datasets/aneurysm/cta_datasets/internal_test/crop_0.4_vessel/Ts0002.nii.gz"

vessels = sitk.ReadImage(path_vessels)

vessels_array = sitk.GetArrayFromImage(vessels)

In [5]:
vessels_array.shape

(425, 615, 615)

In [18]:
path_brain = "/projects/vig/Datasets/aneurysm/cta_datasets/internal_test/crop_0.4_totalseg/Ts0002/brain.nii.gz"  
mask_brain = sitk.ReadImage(path_brain)
mask_brain_array = sitk.GetArrayFromImage(mask_brain)
mask_brain_array.shape

cvs_bbox_path = "/projects/vig/Datasets/aneurysm/cta_datasets/internal_test/cvs_bbox/Ts0002.nii.gz"
cvs_bbox = sitk.ReadImage(cvs_bbox_path)
cvs_bbox_array = sitk.GetArrayFromImage(cvs_bbox)
cvs_bbox_array.shape  

(425, 615, 615)

In [19]:
# combine with or 

combined = np.logical_or(cvs_bbox_array, mask_brain_array)
combined.shape


(425, 615, 615)

In [39]:
# how many 32x32x32 patches can we extract from vessels_array

patch_size = 32
strides = 32
total = 0
total_vessel = 0
total_brain = 0
total_lvl_2 = 0
total_vessel_lvl_2 = 0
for i in range(0, vessels_array.shape[0] - patch_size + 1, strides):
    for j in range(0, vessels_array.shape[1] - patch_size + 1, strides):
        for k in range(0, vessels_array.shape[2] - patch_size + 1, strides):
            total += 1
            patch = vessels_array[i:i+patch_size, j:j+patch_size, k:k+patch_size]
            patch = patch == 1 # artery only
            if np.sum(patch) > 32*32*32/100:  # if more than 1% of the patch is vessel
                total_vessel += 1
                # repeat the three for loops but now inside the patch
                for ii in range(0, patch_size - patch_size//2 + 1, patch_size//2):
                    for jj in range(0, patch_size - patch_size//2 + 1, patch_size//2):
                        for kk in range(0, patch_size - patch_size//2 + 1, patch_size//2):
                            sub_patch = patch[ii:ii+patch_size//2, jj:jj+patch_size//2, kk:kk+patch_size//2]
                            total_lvl_2 += 1
                            if np.sum(sub_patch) > 16*16*16/4: # if more than 25% of the sub-patch is vessel
                                total_vessel_lvl_2 += 1
            if np.sum(combined[i:i+patch_size, j:j+patch_size, k:k+patch_size]) > 0:
                total_brain += 1
            

print(total)
print(total_vessel, total_vessel*8)
print(total_brain)
print(total_lvl_2)
print(total_vessel_lvl_2, total_vessel_lvl_2*8)

4693
170 1360
881
1360
36 288


In [40]:
881+1360+288

2529

In [37]:
4693 + 4693*8 + 4693*8*8 

342589

(425, 615, 615)

In [ ]:
512*512*512/(32*32*32)

4096.0

In [13]:
# how many 16x16x16 patches can we extract from a 32x32x32 patch
patch_size = 16
strides = 16
total = 0
for i in range(0, 32 - patch_size + 1, strides):
    for j in range(0, 32 - patch_size + 1, strides):
        for k in range(0, 32 - patch_size + 1, strides):
            total += 1
print(total)

8


In [8]:
32*32*32

32768